In [ ]:
!pip install -q git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git pandas soundfile
!mkdir -p /content/data
!unzip -q -o /content/chunks.zip -d /content/data
!find /content/data -maxdepth 2 -not -name "*.flac"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.7/930.7 kB 36.4 MB/s eta 0:00:00
/content/data
/content/data/chunks 3 through 8
/content/data/chunks 3 through 8/Acts_04
/content/data/chunks 3 through 8/Acts_05
/content/data/chunks 3 through 8/Acts_03
/content/data/chunks 3 through 8/Acts_07
/content/data/chunks 3 through 8/Acts_08
/content/data/chunks 3 through 8/Acts_06


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv, json, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from ctc_forced_aligner import (
    load_audio, load_alignment_model, generate_emissions,
    preprocess_text, get_alignments, get_spans, postprocess_results,
)

DATA = Path("/content/data")
FULL = Path("/content/full_audio"); FULL.mkdir(exist_ok=True)
LANG = "nus"
MIN_WPS, MAX_WPS = 0.8, 5.0


def build_full_audio(folder):
    parts, rows, pos, sr = [], [], 0, None
    for f in sorted(folder.glob("*.flac")):
        data, rate = sf.read(str(f), dtype="float32")
        if data.ndim > 1: data = data.mean(axis=1)
        sr = sr or rate
        assert rate == sr, f"{f.name} has a different sample rate ({rate})"
        rows.append({"file_name": f.name, "start_s": pos / sr, "end_s": (pos + len(data)) / sr})
        parts.append(data); pos += len(data)
    out = FULL / f"{folder.name}_full.wav"
    sf.write(str(out), np.concatenate(parts), sr)
    return str(out), rows


def clean_transcription(text):
    text = unicodedata.normalize("NFC", text).lower()
    for a in "\u2018\u2019\u02bc`\u00b4": text = text.replace(a, "'")
    # Keep '/' intact. Remove other dash/hyphen variants
    text = re.sub(r"[\u2010-\u2015\-_]", " ", text)
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if ch.isspace(): out.append(" ")
        elif cat[0] in ("L", "M", "N"): out.append(ch)
        elif cat == "Cf" or ch in ("'", "/"): out.append(ch) # Keep slash
        else: out.append(" ")
    return unicodedata.normalize("NFC", re.sub(r"\s+", " ", "".join(out)).strip())


def align_file(audio_path, text, model, tokenizer):
    waveform = load_audio(audio_path, model.dtype, model.device)
    emissions, stride = generate_emissions(model, waveform, batch_size=4)
    tokens_starred, text_starred = preprocess_text(text, romanize=True, language=LANG)
    segments, scores, blank = get_alignments(emissions, tokens_starred, tokenizer)
    spans = get_spans(tokens_starred, segments, blank)
    return postprocess_results(text_starred, spans, stride, scores)


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device)
model, tokenizer = load_alignment_model(device, dtype=dtype)

folders = sorted(p for p in DATA.rglob("Acts_*") if p.is_dir() and re.fullmatch(r"Acts_\d+", p.name))
print(f"{len(folders)} chapter folders found\n")
summary = []

for folder in folders:
    try:
        txts = sorted(folder.glob("*.txt"))
        assert txts, "no .txt transcript"
        text = clean_transcription(txts[0].read_text(encoding="utf-8-sig"))
        audio_path, manifest = build_full_audio(folder)
        words = align_file(audio_path, text, model, tokenizer)
        json.dump(words, open(folder / "words.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

        good, flagged, placed = [], [], 0
        for row in manifest:
            s, e = row["start_s"], row["end_s"]
            ws = [w["text"] for w in words if s <= (w["start"] + w["end"]) / 2 < e]
            placed += len(ws)
            wps = len(ws) / (e - s)
            reason = ("no words" if not ws else
                      f"too few words ({wps:.1f}/s)" if wps < MIN_WPS else
                      f"too many words ({wps:.1f}/s)" if wps > MAX_WPS else None)
            entry = {"file_name": row["file_name"], "transcription": " ".join(ws)}
            (flagged if reason else good).append({**entry, "reason": reason} if reason else entry)

        with open(folder / "metadata.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["file_name", "transcription"]); w.writeheader(); w.writerows(good)
        with open(folder / "review.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["file_name", "transcription", "reason"]); w.writeheader(); w.writerows(flagged)

        summary.append({"chapter": folder.name, "text_words": len(text.split()), "aligned": len(words),
                        "placed": placed, "usable": len(good), "flagged": len(flagged),
                        "mean_score": round(float(np.mean([w["score"] for w in words])), 3)})
        print(f"{folder.name}: usable {len(good)}, flagged {len(flagged)}, mean score {summary[-1]['mean_score']}")
    except Exception as e:
        print(f"{folder.name}: FAILED -> {e}")

df = pd.DataFrame(summary)
df.to_csv(DATA / "alignment_summary.csv", index=False)
print("\n", df.to_string(index=False))

device: cuda


config.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

6 chapter folders found



/usr/local/lib/python3.13/dist-packages/ctc_forced_aligner/alignment_utils.py:133: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:1586.)
  return torch.frombuffer(out, dtype=torch.int16).flatten().to(dtype).to(device) / 32768.0


Acts_03: usable 41, flagged 0, mean score -0.192
Acts_04: usable 48, flagged 0, mean score -0.28
Acts_05: usable 62, flagged 0, mean score -0.239
Acts_06: usable 23, flagged 0, mean score -0.185
Acts_07: usable 81, flagged 0, mean score -0.381
Acts_08: usable 56, flagged 0, mean score -0.276

 chapter  text_words  aligned  placed  usable  flagged  mean_score
Acts_03         618      618     618      41        0      -0.192
Acts_04         863      863     863      48        0      -0.280
Acts_05        1036     1036    1036      62        0      -0.239
Acts_06         387      387     387      23        0      -0.185
Acts_07        1448     1448    1448      81        0      -0.381
Acts_08         937      937     937      56        0      -0.276


In [ ]:
!cd /content/data && zip -qr /content/aligned_results.zip . -i "*.csv" "*.json"
from google.colab import files
files.download("/content/aligned_results.zip")

# backup copy on Google Drive, so a closed session can't wipe it again
from google.colab import drive
drive.mount('/content/drive')
!cp /content/aligned_results.zip /content/drive/MyDrive/aligned_results.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
